In [1]:
!pip -q install biopython lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 46.6 MB/s eta 0:00:00


In [3]:
from Bio import Entrez
import os, json, re, requests, tarfile, zipfile
from pathlib import Path

Entrez.email = "sebbahaya03@gmail.com"

faut utiliser les pmcid parceque ncbi heberge les fichier qu'on a besoin pas le pmids

In [7]:
from pathlib import Path
import requests, re, tarfile, zipfile
from Bio import Entrez

OA_URL = "https://www.ncbi.nlm.nih.gov/pmc/utils/oa/oa.fcgi"

def fetch_ncbi_models(disease_term, out_dir="ncbi_models", retmax=20):
    OUT = Path(out_dir); OUT.mkdir(exist_ok=True)

    #fonction qui cherche la malade et le fichiers sur la BD pubmed
    query = f'({disease_term}) AND (SBML OR SED-ML OR OMEX OR "COMBINE archive" OR "combine archive")'
    search_query = Entrez.read(Entrez.esearch(db="pubmed", term=query, retmax=retmax))
    pmids = search_query["IdList"]  # <- c'est IdList (majuscule)

    print("QUERY:", query)
    print("PMIDs:", pmids)

    pmcids = set() #ca cree un truc qui est vide et set cest pour eviter les doublons
    linkres = Entrez.read(Entrez.elink( #demande pour ce pmcids donne moi les liens
        dbfrom="pubmed", db="pmc", id=",".join(pmids), linkname="pubmed_pmc"))  #transforme la réponse XML en structure Python (dict/list)

    for link in linkres: #link cest lien et linkre cest le bloc de res
        dbs = link.get("LinkSetDb", []) #si la clé n’existe pas on met une liste vide (évite crash)
        if dbs and dbs[0].get("Link"):
            pmcids.add("PMC" + str(dbs[0]["Link"][0]["Id"]))# on ajoute le id au set, on doit le transformer en PMCIDS

    pmcids = sorted(pmcids) #trié histoire d'avoir un affichage propre
    print("PMCIDs:", len(pmcids)) #on affiche combien y'a d'articles

    if not pmcids:
        return {"omex": [], "sedml": [], "sbml_xml": []}

    def oa_link(pmcid): #prends le pmcid et renvoie l'url
        resp = requests.get(OA_URL, params={"id": pmcid}, timeout=30) #service officiel PMC OA
        if resp.status_code != 200: #répond avec une page (XML/texte) qui contient des liens vers les fichiers téléchargeables
            return None
        hrefs = re.findall(r'href="([^"]+)"', resp.text) #extrait toutes les URL présentes dans la réponse
        for u in hrefs:
            if u.endswith(".tar.gz") or u.endswith(".tgz"): #priorité aux archives tar.gz / tgz (souvent le format principal des packages OA)
                # Convert ftp:// to https:// if found
                if u.startswith("ftp://"):
                    u = u.replace("ftp://", "https://")
                return u
        for u in hrefs:
            if u.endswith(".zip"): #Si on n’a pas trouvé de tar.gz, on cherche un .zip et on renvoie le premier trouvé
                # Convert ftp:// to https:// if found
                if u.startswith("ftp://"):
                    u = u.replace("ftp://", "https://")
                return u
        return None

    def extract(archive_path, dest): #décompresser l’archive téléchargée
        dest.mkdir(parents=True, exist_ok=True)
        p = str(archive_path).lower() #convertit le chemin en texte, en minuscules,comme ça on peut tester l’extension facilement
        if p.endswith(".zip"): # force a dezipper
            with zipfile.ZipFile(archive_path, "r") as z:
                z.extractall(dest)
        else: #le lire comme meme
            with tarfile.open(archive_path, "r:gz") as t:
                t.extractall(dest, filter='data') # Added filter='data' here

    def is_sbml_xml(path):  #test si cest vraiment un fichier xml (sbml) pas xml
        try:
            head = open(path, "rb").read(3000).lower()  # <- on lit les bytes
            return b"<sbml" in head #cest pour verifier que la balise est une sbml et pas xml
        except:
            return False

    kept_files = {"omex": [], "sedml": [], "sbml_xml": []}
    for pmcid in pmcids:
        link = oa_link(pmcid) # ca appelle la fonction oa_link (pas récursivité)
        if not link:
            continue

        art_dir = OUT / pmcid
        art_dir.mkdir(exist_ok=True)
        arch = art_dir / Path(link).name #le chemin du fichier archive qu’on va télécharger dans ce dossier.

        if not arch.exists():
            with requests.get(link, stream=True, timeout=60) as rr:
                rr.raise_for_status()
                with open(arch, "wb") as f:
                    for chunk in rr.iter_content(chunk_size=1024*256):
                        if chunk:
                            f.write(chunk)

        exdir = art_dir / "extracted"
        extract(arch, exdir)

        omex = [str(p) for p in exdir.rglob("*.omex")]
        sedml = [str(p) for p in exdir.rglob("*.sedml")]
        sbml = [str(p) for p in exdir.rglob("*.xml") if is_sbml_xml(p)]

        # garder seulement les fichier qu'on a vraiment besoin
        kept_files["omex"] += omex
        kept_files["sedml"] += sedml
        kept_files["sbml_xml"] += sbml

        if omex or sedml or sbml:
            print("FOUND", pmcid, "| omex", len(omex), "| sedml", len(sedml), "| sbml", len(sbml)) #pour faire joli

    print("TOTAL omex:", len(kept_files["omex"]))
    print("TOTAL sedml:", len(kept_files["sedml"]))
    print("TOTAL sbml:", len(kept_files["sbml_xml"]))
    print("Saved in:", OUT.resolve())
    return kept_files

kept_dengue = fetch_ncbi_models('(dengue OR DENV)', out_dir="dengue_files") # Corrected extra parenthesis
kept_chikungunya = fetch_ncbi_models('(chikungunya OR CHIKV)', out_dir="chikungunya_files")
kept_lyme = fetch_ncbi_models('(lyme OR borrelia OR borreliosis)', out_dir="lyme_files")
kept_mpox = fetch_ncbi_models('(mpox OR monkeypox)', out_dir="mpox_files")
kept_west_nile = fetch_ncbi_models('("west nile" OR WNV)', out_dir="west_nile_files")
kept_influenza = fetch_ncbi_models('(influenza OR "influenza virus" OR "avian influenza" OR H5N1)', out_dir="influenza_files")
kept_tuberculosis = fetch_ncbi_models('(tuberculosis OR TB OR mycobacterium)', out_dir="tuberculosis_files")
kept_hiv = fetch_ncbi_models('(HIV OR "human immunodeficiency virus")', out_dir="hiv_files")
kept_covid = fetch_ncbi_models('("covid" OR "SARS-CoV-2")', out_dir="covid_files")

QUERY: ((dengue OR DENV) )) AND (SBML OR SED-ML OR OMEX OR "COMBINE archive" OR "combine archive")
PMIDs: []
PMCIDs: 0
QUERY: ((chikungunya OR CHIKV)) AND (SBML OR SED-ML OR OMEX OR "COMBINE archive" OR "combine archive")
PMIDs: []
PMCIDs: 0
QUERY: ((lyme OR borrelia OR borreliosis)) AND (SBML OR SED-ML OR OMEX OR "COMBINE archive" OR "combine archive")
PMIDs: []
PMCIDs: 0
QUERY: ((mpox OR monkeypox)) AND (SBML OR SED-ML OR OMEX OR "COMBINE archive" OR "combine archive")
PMIDs: []
PMCIDs: 0
QUERY: (("west nile" OR WNV)) AND (SBML OR SED-ML OR OMEX OR "COMBINE archive" OR "combine archive")
PMIDs: []
PMCIDs: 0
QUERY: ((influenza OR "influenza virus" OR "avian influenza" OR H5N1)) AND (SBML OR SED-ML OR OMEX OR "COMBINE archive" OR "combine archive")
PMIDs: ['31333719', '24088197']
PMCIDs: 1


/tmp/ipykernel_278/2263818900.py:60: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall(dest)


TOTAL omex: 0
TOTAL sedml: 0
TOTAL sbml: 0
Saved in: /content/influenza_files
QUERY: ((tuberculosis OR TB OR mycobacterium)) AND (SBML OR SED-ML OR OMEX OR "COMBINE archive" OR "combine archive")
PMIDs: ['21609434']
PMCIDs: 1
TOTAL omex: 0
TOTAL sedml: 0
TOTAL sbml: 0
Saved in: /content/tuberculosis_files
QUERY: ((HIV OR "human immunodeficiency virus")) AND (SBML OR SED-ML OR OMEX OR "COMBINE archive" OR "combine archive")
PMIDs: []
PMCIDs: 0
QUERY: (("covid" OR "SARS-CoV-2")) AND (SBML OR SED-ML OR OMEX OR "COMBINE archive" OR "combine archive")
PMIDs: ['34986226', '34799364', '34041012']
PMCIDs: 1
TOTAL omex: 0
TOTAL sedml: 0
TOTAL sbml: 0
Saved in: /content/covid_files
